# Messaging & Event-Driven Architectures

## What's covered

- **Why asynchronous communication exists** — the problems it solves over synchronous calls
- **Message queues** — the basic primitive and its three delivery guarantees
- **Pub/sub** — fan-out beyond point-to-point queues
- **Event-driven architecture** as a system shape, not just a transport
- **Event sourcing** — making state a derivable function of an append-only log
- **CQRS** — splitting read and write paths so each can scale independently
- The dominant implementations — Kafka, RabbitMQ, SQS, Pulsar — and what each fits


## Why asynchronous communication

Synchronous calls — `request → response` over HTTP, gRPC, or a direct database call — are the easy default. Caller blocks, server processes, response comes back. The dependency graph is explicit and easy to reason about.

It breaks down at scale in three predictable ways:

- **Coupling.** Service A directly calls service B which calls C which calls D. If D is slow or down, A is slow or down. Latency budgets compound. Failures cascade.
- **Back-pressure.** A spike in incoming load floods downstream services that can't keep up. The whole chain slows or fails together, often with retry storms making it worse.
- **Fan-out.** When one event needs to trigger five different reactions, synchronous orchestration means the originating call waits for all five to finish.

Asynchronous messaging breaks the synchronous coupling. The producer drops a message on a queue or a topic and moves on. Consumers pick up messages at their own pace. A slow consumer falls behind; the queue absorbs the burst. A failed consumer eventually catches up after restarting. The producer doesn't know — or care — who's consuming.

The trade-off is real: you lose immediacy (the producer doesn't know when, or whether, the consumer succeeded), and you take on the operational burden of running a message broker. Async is the right tool when **the work can wait**, when **multiple consumers care about the same event**, or when **the rate of incoming work doesn't match the rate of processing**. It's overkill when a simple synchronous call suffices.


## Message queues — the basic primitive

A **message queue** is an ordered, durable buffer of messages. Producers append; consumers read. The queue lives on a broker (a dedicated service: Kafka, RabbitMQ, SQS, Pulsar) that handles persistence, replication, and delivery.

The defining property of a queue (as opposed to a topic) is **point-to-point delivery** — each message is processed by exactly one consumer. Multiple consumers can read from the same queue; the broker distributes messages across them, *not* duplicating.

**The three delivery guarantees** every messaging system has to choose among:

### At-most-once

The broker delivers each message zero or one times. Lost messages stay lost. Cheapest. Used when occasional data loss is acceptable (metrics, debug logs, click events for aggregate analytics).

### At-least-once

The broker delivers each message one or more times. No message is lost, but some might be delivered twice (or more) due to consumer crashes between processing and acknowledging. The consumer must be **idempotent** — processing the same message twice must produce the same result as processing it once.

This is the most common guarantee in production. Almost every modern broker offers it as the default. The trade-off — duplicates instead of losses — is usually the right one, because idempotency is achievable in code (deduplication keys, idempotent operations) while data loss is not recoverable.

### Exactly-once

The broker delivers each message exactly once, no duplicates, no losses. The grail.

In a strict sense, exactly-once delivery is impossible across an unreliable network (you can't tell whether your "ack" was lost or the message was never received without an extra round-trip). In practice, modern brokers approximate it with **transactional writes**: the consumer's processing and its acknowledgment are wrapped in a transaction that either both succeed or both fail. Kafka's "exactly-once semantics" works this way, with significant performance cost.

For most workloads, **at-least-once + idempotent consumers** is the practical sweet spot — same end-to-end guarantees as exactly-once, no broker-side complexity.


### At-least-once with idempotent consumer


In [ ]:
# Simulate at-least-once delivery and the idempotent consumer pattern.

import uuid
from dataclasses import dataclass, field


@dataclass
class Message:
    id: str            # unique message ID — set by producer or broker
    payload: dict


@dataclass
class Broker:
    queue: list[Message] = field(default_factory=list)

    def publish(self, msg: Message) -> None:
        self.queue.append(msg)

    def deliver(self) -> Message | None:
        if not self.queue:
            return None
        return self.queue[0]    # peek — message stays until ack

    def ack(self, msg_id: str) -> None:
        self.queue = [m for m in self.queue if m.id != msg_id]


# A naive consumer — credits funds twice if the message is redelivered.
class NaiveConsumer:
    def __init__(self):
        self.balances: dict[str, int] = {}

    def process(self, msg: Message) -> None:
        user = msg.payload["user"]
        self.balances[user] = self.balances.get(user, 0) + msg.payload["amount"]


# An idempotent consumer — tracks processed message IDs.
class IdempotentConsumer:
    def __init__(self):
        self.balances: dict[str, int] = {}
        self.processed: set[str] = set()

    def process(self, msg: Message) -> None:
        if msg.id in self.processed:
            return                              # already handled; safe to skip
        user = msg.payload["user"]
        self.balances[user] = self.balances.get(user, 0) + msg.payload["amount"]
        self.processed.add(msg.id)


# Simulate the at-least-once redelivery scenario:
# consumer processes, then crashes before acking, so the message is redelivered.
msg = Message(id=str(uuid.uuid4()), payload={"user": "alice", "amount": 100})

naive = NaiveConsumer()
naive.process(msg)              # first delivery
naive.process(msg)              # redelivery after a crash
print(f"naive consumer balances:      {naive.balances}")
# alice ended up with 200 — the redelivery double-credited her.

idempotent = IdempotentConsumer()
idempotent.process(msg)
idempotent.process(msg)         # redelivery — second processing is a no-op
print(f"idempotent consumer balances: {idempotent.balances}")
# alice has 100, regardless of how many times the message arrived.


The pattern generalizes: **store the processed message ID alongside the state change, in the same transaction**. On redelivery, the consumer sees the ID was already processed and short-circuits. The deduplication store can be a database table, Redis, or a built-in feature of the broker (Kafka's consumer offsets work this way).


## Pub/Sub — fan-out beyond queues

A queue gives you point-to-point delivery — each message goes to *one* consumer. **Pub/sub** (publish-subscribe) gives you fan-out — each message goes to *every* subscriber.

The vocabulary shifts. Producers publish to **topics**. Consumers subscribe to topics. The broker holds one logical message per publication and delivers it to every subscriber.

**Two flavours in modern systems:**

- **Broadcast topics.** Every subscriber gets every message. Used for events that any number of services might care about — "user signed up", "order completed", "deploy started."
- **Consumer groups.** A topic has multiple subscriber *groups*; within a group, messages are partitioned across instances (like a queue), but between groups, every message is delivered to each group. Lets one event feed both a "send welcome email" service (one instance per group consumes one event) and an "update analytics" service (its own group, also consumes every event). Kafka, Pulsar, and Kinesis all use this model.

**Pub/sub vs message queue** is often a false dichotomy in modern brokers. Kafka topics with one consumer group behave like queues; with multiple groups behave like pub/sub. RabbitMQ has both queues and exchanges (its pub/sub primitive) and lets you compose them. SQS is queue-only; SNS is pub/sub; AWS users commonly fan out SNS to multiple SQS queues to get both behaviours.

**The decoupling win** is the real reason to reach for pub/sub. The producer doesn't know what services consume its events. New consumers can be added without touching the producer. Old consumers can be removed without coordination. The topic is the contract, and the contract is just "events of shape X arrive here."


## Event-driven architecture as a system shape

A messaging system can be a **transport** — a way to ship work from A to B asynchronously — or it can be the **organizing principle of the whole system**. The latter is event-driven architecture (EDA).

In an event-driven system, **events are the source of truth**. When something happens in the domain (`OrderPlaced`, `PaymentReceived`, `InventoryReserved`), the service responsible publishes an event. Other services react to it. There's no orchestrating coordinator; the dependency graph is implicit in who subscribes to what.

**Two flavours:**

- **Event notification.** The event is a thin "something happened" signal. Subscribers fetch details from the producer if they need them. Loose coupling at the cost of follow-up reads.
- **Event-carried state transfer.** The event carries enough detail that subscribers can act without fetching anything else. Tighter coupling to the event schema, but faster reactions and resilience to producer downtime.

**The wins** of event-driven over orchestrated synchronous flows:

- **Loose coupling.** Producers don't know consumers.
- **Resilience.** A slow or down consumer doesn't break the producer; events queue up and are processed when the consumer recovers.
- **Auditability.** The event log is a record of everything that happened in the domain — extremely valuable for debugging, compliance, and replaying state.
- **Extensibility.** New consumers can be added without touching existing services.

**The costs:**

- **Eventual consistency by construction.** A subscriber's view of the world lags the producer by however long event delivery and processing takes. Application code must be written to expect this.
- **Debugging is harder.** A single user action triggers a chain of events that fans across services. Distributed tracing becomes essential (we cover this in notebook 07).
- **Schema evolution is harder.** Events outlive code. An event in your log written six months ago must still be readable today. Schema-registry tools (Confluent, Apicurio) and versioning conventions exist to manage this.

**When to reach for event-driven.** When the domain is naturally event-shaped — things happen, multiple parties care, the chain of reactions is open-ended. E-commerce flows, supply chains, financial transactions, multi-step workflows. **When not to.** Simple CRUD apps. Two-service systems. Anywhere a synchronous request fits the mental model of the user action.


## Event sourcing — state as a function of events

Most systems store **current state** in a database. The user's address is `"123 Main St"` because that's what's in the `users.address` column. If they update it, the column is overwritten; the old value is lost (unless a separate audit log captures it).

**Event sourcing inverts this.** The system stores the *sequence of events* that happened. State is *derived* by replaying the events. The user's address is the result of folding `UserCreated → AddressChanged(123 Main St)` over time; if they update it, a new `AddressChanged` event is appended, and the derived state changes.

```
  traditional:   state = users.address    (overwritten on every update)

  event-sourced: state = fold(events from start)
                 events = [UserCreated, AddressChanged, AddressChanged, ...]
                 (events are append-only; nothing is ever overwritten)
```

**The benefits are real:**

- **Full audit trail.** Every change is preserved. You can answer "what did the system look like on March 5th?" by replaying events up to that date.
- **Temporal queries.** "How many times did this user change their email?" is just a count over the event log.
- **Bug fix replay.** A bug in projection logic that miscalculated balances for two months can be fixed by replaying events through the corrected logic — the source of truth is intact.
- **Multiple projections.** The same events can produce different views — a normalized SQL table for transactional reads, a denormalized search index for full-text, a graph database for relationship queries. Each is a *projection*.

**The costs:**

- **Querying current state is no longer a simple SELECT.** You need a materialized view (a projection) of the events into a queryable shape. Maintaining projections is its own subsystem.
- **Schema evolution is harder.** Old events must remain readable forever. You can't just drop a field.
- **The mental model is different.** Developers used to "write current state, read current state" need to learn "publish events, subscribe to events, build projections."

Event sourcing pairs naturally with event-driven architecture but is independent of it. You can use event sourcing inside a single service (just for that service's local state) without exposing events to others.


## CQRS — splitting reads and writes

CQRS — **Command Query Responsibility Segregation** — is the architectural pattern of separating the **write side** (commands that change state) from the **read side** (queries that observe state) into different models, often different data stores.

```
                    +-----------+
   client write --> | commands  |--> write model --> persist
                    +-----------+                       |
                                                        | publishes events
                                                        v
                                                  +-----------+
                                                  | projector |--> read model (denormalized)
                                                  +-----------+              |
                                                                             v
                                                                        +---------+
                                                       client read <--- | queries |
                                                                        +---------+
```

The motivation: reads and writes have *very different requirements* in many systems.

- Writes need **strong consistency and transactional integrity**, but are usually low-volume.
- Reads are typically **much higher volume**, and need to be **fast**, but can tolerate eventual consistency.

If you serve both from the same normalized relational schema, you're optimizing for neither. CQRS lets you write to a strongly consistent transactional store and *project* to multiple read-optimized stores — denormalized SQL views, search indexes, graph stores, in-memory caches — each tuned for a specific query pattern.

**CQRS pairs especially well with event sourcing** because the event log is a natural source for projections. Each projection is a consumer of the event log, building its own read-optimized view. Adding a new query pattern is adding a new projection; rebuilding a projection is replaying the events.

**The trade-offs:**

- **Reads are now eventually consistent.** A write that just succeeded may not be visible on the read side for some milliseconds-to-seconds. The application must be designed for this.
- **More moving parts.** Projections to maintain, event log to operate, dual paths to test.
- **Operational complexity.** Catching up a projection from cold can take hours for a high-volume system; you need to think about projection bootstrapping carefully.

CQRS without event sourcing is also possible — write to a normalized DB, project to denormalized views via change-data-capture. This is the more common variant; full event-sourced CQRS is reserved for systems that genuinely benefit from the event log as the source of truth.


## The dominant implementations

| Broker | Style | Strengths | Weaknesses |
|---|---|---|---|
| **Kafka** | Distributed commit log | High throughput, durable, replayable, ecosystem (Connect, Streams), good for event sourcing | Operational complexity, partition-based ordering only |
| **RabbitMQ** | AMQP queues + exchanges | Flexible routing (topics, headers, fan-out), low-latency point-to-point | Lower per-broker throughput, message store can grow without bound |
| **AWS SQS** | Managed queue | Zero ops, scales automatically, integrates with Lambda | At-most-1MB messages, no built-in topic-style fan-out (use SNS), at-least-once only |
| **AWS SNS** | Managed pub/sub | Fan-out, integrates with SQS/Lambda/HTTP | No ordering guarantees in standard topics |
| **Pulsar** | Hybrid log + queue | Geo-replication, multi-tenancy, decoupled compute and storage | Smaller ecosystem than Kafka |
| **Google Pub/Sub** | Managed pub/sub | Global ordering, exactly-once support, Spanner-backed | GCP-only |
| **Redis Streams** | In-memory log | Sub-ms latency, simple ops | Less durable than disk-backed brokers; bounded by RAM |

**Picking one.** For event-sourced systems and high-throughput pipelines, Kafka or Pulsar. For traditional task queues and rich routing, RabbitMQ or SQS. For managed everything in a cloud, SQS+SNS on AWS or Pub/Sub on GCP. For sub-millisecond fan-out within a single cluster, Redis Streams.

**The most common mistake** is picking the most powerful broker for a workload that doesn't need it. Kafka is overkill for a service-to-service task queue with a few thousand messages per second; SQS would be a much better fit operationally. Match the broker's complexity to the problem's complexity.


Notebook six turns to **Microservices & Resilience** — the architectural style most large systems land in, and the resilience patterns that keep it standing. Monolith vs. microservices and the real trade-offs. API gateway and service mesh as the wiring layer. The saga pattern for transactions that span services. And the resilience trio — circuit breaker, bulkhead, idempotent retries — that lets a microservice mesh stay up when its parts don't.
